Otetaan koko tiedostosta (US_Accidents_March23.csv) 100 000 erä alkukartoitusta varten. Tarkoituksena on luoda yleiskuva aineistosta. Tallennetaan istunnon ajalle erä (sample_df) ja tulostetaan schema, joka näyttää myös columnien nimet. 

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Big").getOrCreate()

df = spark.read.csv("C:/Users/serko/US_Accidents/US_Accidents_March23.csv", header=True, inferSchema=True)

sample_df = df.limit(100000)
sample_df.cache()
sample_df.printSchema()


root
 |-- ID: string (nullable = true)
 |-- Source: string (nullable = true)
 |-- Severity: integer (nullable = true)
 |-- Start_Time: timestamp (nullable = true)
 |-- End_Time: timestamp (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)
 |-- End_Lat: double (nullable = true)
 |-- End_Lng: double (nullable = true)
 |-- Distance(mi): double (nullable = true)
 |-- Description: string (nullable = true)
 |-- Street: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Timezone: string (nullable = true)
 |-- Airport_Code: string (nullable = true)
 |-- Weather_Timestamp: timestamp (nullable = true)
 |-- Temperature(F): double (nullable = true)
 |-- Wind_Chill(F): double (nullable = true)
 |-- Humidity(%): double (nullable = true)
 |-- Pressure(in): double (nullable = true)
 |-- V

In [ ]:
# tulostetaan näyte-erän 10 ensimmäistä riviä

sample_df.show(10)

+----+-------+--------+-------------------+-------------------+------------------+------------------+-------+-------+------------+--------------------+--------------------+------------+----------+-----+----------+-------+----------+------------+-------------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+-----------------+-------+-----+--------+--------+--------+-------+-------+----------+-------+-----+---------------+--------------+------------+--------------+--------------+-----------------+---------------------+
|  ID| Source|Severity|         Start_Time|           End_Time|         Start_Lat|         Start_Lng|End_Lat|End_Lng|Distance(mi)|         Description|              Street|        City|    County|State|   Zipcode|Country|  Timezone|Airport_Code|  Weather_Timestamp|Temperature(F)|Wind_Chill(F)|Humidity(%)|Pressure(in)|Visibility(mi)|Wind_Direction|Wind_Speed(mph)|Precipitation(in)|Weather_Condition|A

Tarkastellaan puuttuvia arvoja sarakekohtaisesti ja niiden prosenttiosuuksia niiden columnien osalta, joissa arvo muu kuin 0

In [3]:
from pyspark.sql.functions import col, sum

null_counts = sample_df.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in sample_df.columns
])

null_counts.show()


+---+------+--------+----------+--------+---------+---------+-------+-------+------------+-----------+------+----+------+-----+-------+-------+--------+------------+-----------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+-----------------+-------+----+--------+--------+--------+-------+-------+----------+-------+----+---------------+--------------+------------+--------------+--------------+-----------------+---------------------+
| ID|Source|Severity|Start_Time|End_Time|Start_Lat|Start_Lng|End_Lat|End_Lng|Distance(mi)|Description|Street|City|County|State|Zipcode|Country|Timezone|Airport_Code|Weather_Timestamp|Temperature(F)|Wind_Chill(F)|Humidity(%)|Pressure(in)|Visibility(mi)|Wind_Direction|Wind_Speed(mph)|Precipitation(in)|Weather_Condition|Amenity|Bump|Crossing|Give_Way|Junction|No_Exit|Railway|Roundabout|Station|Stop|Traffic_Calming|Traffic_Signal|Turning_Loop|Sunrise_Sunset|Civil_Twilight|Nautical_Twil

In [6]:
from pyspark.sql.functions import col, sum, round

# Sarakkeet, joissa ei arvoa
cols_to_check = ["City", "Zipcode", "Timezone", "Airport_Code", "Weather_Timestamp", "Temperature(F)",
                 "Wind_Chill(F)", "Humidity(%)", "Pressure(in)", "Visibility(mi)", "Wind_Direction",
                 "Wind_Speed(mph)", "Precipitation(in)", "Weather_Condition", 
                 "Sunrise_Sunset", "Civil_Twilight", "Nautical_Twilight", "Astronomical_Twilight"]

# Laske puuttuvat arvot
null_counts = sample_df.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in cols_to_check
])

# Laske prosentit
total_rows = sample_df.count()

null_percentages = null_counts.select([
    round((col(c) / total_rows * 100), 3).alias(c) for c in cols_to_check
])

null_percentages.show()


+-----+-------+--------+------------+-----------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+-----------------+--------------+--------------+-----------------+---------------------+
| City|Zipcode|Timezone|Airport_Code|Weather_Timestamp|Temperature(F)|Wind_Chill(F)|Humidity(%)|Pressure(in)|Visibility(mi)|Wind_Direction|Wind_Speed(mph)|Precipitation(in)|Weather_Condition|Sunrise_Sunset|Civil_Twilight|Nautical_Twilight|Astronomical_Twilight|
+-----+-------+--------+------------+-----------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+-----------------+--------------+--------------+-----------------+---------------------+
|0.001|  0.007|   0.007|       0.007|            1.054|         1.591|       95.678|      1.856|       1.292|         1.846|         1.064|          23.82|           92.632|            1.604|         0.001|        

Havainto: Wind_Chill(F), Precipitation(in) puuttuvat arvot yli 90 %, Wind_Speed(mph) liki 25 %, tarkistettava myös koko aineiston osalta

Ero onnettomuuden alkamisajankohdan ja weather timestampin välillä, 20 ensimmäistä riviä antaa yleissilmäyksen, lisäksi laskettu suurin havaittu ero

In [10]:
from pyspark.sql.functions import col, unix_timestamp, abs, round


df_ero = sample_df.withColumn(
    "weather_time_diff_minutes", round(
    abs(unix_timestamp(col("Start_Time")) - unix_timestamp(col("Weather_Timestamp"))) / 60
))

# Näytä ensimmäiset 20 riviä
df_ero.select("Start_Time", "Weather_Timestamp", "weather_time_diff_minutes") \
    .show(20, truncate=False)

# Näytä tilastot
df_ero.select("weather_time_diff_minutes").describe().show()

+-------------------+-------------------+-------------------------+
|Start_Time         |Weather_Timestamp  |weather_time_diff_minutes|
+-------------------+-------------------+-------------------------+
|2016-02-08 05:46:00|2016-02-08 05:58:00|12.0                     |
|2016-02-08 06:07:59|2016-02-08 05:51:00|17.0                     |
|2016-02-08 06:49:27|2016-02-08 06:56:00|7.0                      |
|2016-02-08 07:23:34|2016-02-08 07:38:00|14.0                     |
|2016-02-08 07:39:07|2016-02-08 07:53:00|14.0                     |
|2016-02-08 07:44:26|2016-02-08 07:51:00|7.0                      |
|2016-02-08 07:59:35|2016-02-08 07:56:00|4.0                      |
|2016-02-08 07:59:58|2016-02-08 07:56:00|4.0                      |
|2016-02-08 08:00:40|2016-02-08 07:58:00|3.0                      |
|2016-02-08 08:10:04|2016-02-08 08:28:00|18.0                     |
|2016-02-08 08:14:42|2016-02-08 07:50:00|25.0                     |
|2016-02-08 08:21:27|2016-02-08 08:28:00|7.0    